In [1]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from sentence_transformers import SentenceTransformer

base_model = SentenceTransformer('all-MiniLM-L6-v2')
base_model.similarity

<function sentence_transformers.util.similarity.cos_sim(a: 'list | np.ndarray | Tensor', b: 'list | np.ndarray | Tensor') -> 'Tensor'>

In [ ]:
import torch
import numpy as np
from torch import Tensor
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# 1. Custom Binary Model using Inheritance
# This class inherits from SentenceTransformer and modifies the encode method.
class UBinarySentenceTransformer(SentenceTransformer):
    """
    A SentenceTransformer model that always outputs binary embeddings.
    """
    def encode(self, sentences, *args, **kwargs) -> Tensor:
        """
        Overrides the default encode method to enforce binary precision.
        """
        # Set the desired arguments for binary embeddings
        kwargs['precision'] = "ubinary"
        kwargs['convert_to_tensor'] = True
        
        # Call the original encode method from the parent class (SentenceTransformer)
        return super().encode(sentences, *args, **kwargs)

# 2. Hamming Similarity Function (no changes needed)
def hamming_similarity_from_distance(a: Tensor, b: Tensor) -> Tensor:
    """
    Computes Hamming similarity based on Hamming distance for packed binary tensors.
    This version is more efficient as it avoids unpacking the original tensors.
    Similarity = Total Bits - Differing Bits.
    """
    # Ensure tensors are on the CPU and are of type uint8 for bitwise operations
    a_cpu = a.cpu().to(torch.uint8)
    b_cpu = b.cpu().to(torch.uint8)

    # 1. Use broadcasting to efficiently perform a bitwise XOR.
    # This finds the differing bits between all pairs of vectors.
    # The resulting tensor `differences` has a shape of:
    # (num_queries, num_corpus_docs, embedding_dim_in_bytes)
    differences = a_cpu.unsqueeze(1) ^ b_cpu.unsqueeze(0)

    # 2. Count the number of set bits (1s) to get the Hamming distance.
    # We convert the tensor of byte differences to a NumPy array,
    # unpack the bits for each byte, and sum them up. This is an
    # efficient way to perform a "population count".
    # The result is a matrix of distances with shape: (num_queries, num_corpus_docs)
    hamming_distance = np.unpackbits(differences.numpy(), axis=2).sum(axis=2)
    
    # 3. The total number of bits is the embedding dimension.
    # We get this by taking the number of bytes and multiplying by 8.
    vector_length = a_cpu.shape[1] * 8

    # 4. Convert Hamming distance to Hamming similarity.
    hamming_similarity = vector_length - hamming_distance

    # 5. Return the result as a PyTorch float tensor.
    return torch.from_numpy(hamming_similarity).float()



if __name__ == "__main__":
    # --- Dummy Data ---
    queries = { 'q1': 'what is a transformer', 'q2': 'how to train a model' }
    corpus = { 'c1': 'A transformer is a deep learning model.', 'c2': 'You can train a model using PyTorch.' }
    relevant_docs = { 'q1': {'c1'}, 'q2': {'c2'} }

    # --- Model Setup ---
    # Instantiate our new custom class directly. No wrapper needed.
    model = UBinarySentenceTransformer('all-MiniLM-L6-v2')

    # --- Evaluator Setup ---
    # The evaluator now takes our custom model and the custom scoring function.
    evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        score_functions={"hamming": hamming_similarity_from_distance},
        name='binary-ir-evaluation',

    )

    # --- Run Evaluation ---
    results = evaluator(model, output_path='./')

    print("\nEvaluation successfully completed!")
    print("Results:")
    print(results)

2025-08-28 16:04:47,985 - INFO - Loading Corpus...


  0%|          | 0/3633 [00:00<?, ?it/s]

2025-08-28 16:04:48,011 - INFO - Loaded 3633 TEST Documents.
2025-08-28 16:04:48,011 - INFO - Doc Example: {'text': 'Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 years) 

In [3]:
import torch
import numpy as np
from torch import Tensor
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import logging
import os
import heapq
import json
from tqdm import trange
from custum_evals import MultiGPUInformationRetrievalEvaluator, UBinarySentenceTransformer, hamming_similarity_from_distance
# --- FIX ENDS HERE ---


logger = logging.getLogger(__name__)

# The main execution logic should be wrapped in this block to ensure
# it only runs in the main process, not in the spawned child processes.
if __name__ == '__main__':
    # --- Dummy Data ---
    queries = { 'q1': 'what is a transformer', 'q2': 'how to train a model' }
    corpus = { 'c1': 'A transformer is a deep learning model.', 'c2': 'You can train a model using PyTorch.' }
    relevant_docs = { 'q1': {'c1'}, 'q2': {'c2'} }

    model = UBinarySentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # --- Evaluator Setup ---
    # The evaluator now takes our custom model and the custom scoring function.
    ks = [1, 2] # Adjusted ks to be <= number of corpus docs
    batch_size = 2
    evaluator = MultiGPUInformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        score_functions={"hamming": hamming_similarity_from_distance},
        name='binary-ir-evaluation',
        batch_size=batch_size,
        mrr_at_k=ks,
        ndcg_at_k=ks,
        accuracy_at_k=ks,
        precision_recall_at_k=ks,
        map_at_k=ks,
        show_progress_bar=True,
        write_csv=True,
        encode_chunk_size=1024, 
        encode_batch_size=batch_size,
    )

    # --- Run Evaluation ---
    results = evaluator(model, output_path='./')

    print("\nEvaluation successfully completed!")
    print("Results:")
    print(results)

No sentence-transformers model found with name sentence-transformers/all-MiniLM-L6-v2. Creating a new one with mean pooling.


Computing query embeddings


Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Computing corpus embeddings


Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00, 2078.45it/s]


Evaluation successfully completed!
Results:
{'binary-ir-evaluation_hamming_accuracy@1': 1.0, 'binary-ir-evaluation_hamming_accuracy@2': 1.0, 'binary-ir-evaluation_hamming_precision@1': 1.0, 'binary-ir-evaluation_hamming_precision@2': 0.5, 'binary-ir-evaluation_hamming_recall@1': 1.0, 'binary-ir-evaluation_hamming_recall@2': 1.0, 'binary-ir-evaluation_hamming_ndcg@1': 1.0, 'binary-ir-evaluation_hamming_ndcg@2': 1.0, 'binary-ir-evaluation_hamming_mrr@1': 1.0, 'binary-ir-evaluation_hamming_mrr@2': 1.0, 'binary-ir-evaluation_hamming_map@1': 1.0, 'binary-ir-evaluation_hamming_map@2': 1.0}


In [47]:
embeddings

tensor([[ 6.7657e-02,  6.3496e-02,  4.8713e-02,  7.9305e-02,  3.7448e-02,
          2.6528e-03,  3.9375e-02, -7.0984e-03,  5.9361e-02,  3.1537e-02,
          6.0098e-02, -5.2905e-02,  4.0607e-02, -2.5931e-02,  2.9843e-02,
          1.1269e-03,  7.3515e-02, -5.0382e-02, -1.2239e-01,  2.3703e-02,
          2.9727e-02,  4.2477e-02,  2.5634e-02,  1.9952e-03, -5.6919e-02,
         -2.7160e-02, -3.2904e-02,  6.6025e-02,  1.1901e-01, -4.5879e-02,
         -7.2621e-02, -3.2584e-02,  5.2341e-02,  4.5055e-02,  8.2530e-03,
          3.6702e-02, -1.3942e-02,  6.5392e-02, -2.6427e-02,  2.0638e-04,
         -1.3664e-02, -3.6281e-02, -1.9504e-02, -2.8974e-02,  3.9427e-02,
         -8.8409e-02,  2.6243e-03,  1.3671e-02,  4.8306e-02, -3.1157e-02,
         -1.1733e-01, -5.1169e-02, -8.8529e-02, -2.1896e-02,  1.4299e-02,
          4.4417e-02, -1.3482e-02,  7.4339e-02,  2.6638e-02, -1.9876e-02,
          1.7919e-02, -1.0605e-02, -9.0426e-02,  2.1327e-02,  1.4120e-01,
         -6.4717e-03, -1.4039e-03, -1.

In [48]:
embeddings2

tensor([[ 6.7657e-02,  6.3496e-02,  4.8713e-02,  7.9305e-02,  3.7448e-02,
          2.6528e-03,  3.9375e-02, -7.0984e-03,  5.9361e-02,  3.1537e-02,
          6.0098e-02, -5.2905e-02,  4.0607e-02, -2.5931e-02,  2.9843e-02,
          1.1269e-03,  7.3515e-02, -5.0382e-02, -1.2239e-01,  2.3703e-02,
          2.9727e-02,  4.2477e-02,  2.5634e-02,  1.9952e-03, -5.6919e-02,
         -2.7160e-02, -3.2904e-02,  6.6025e-02,  1.1901e-01, -4.5879e-02,
         -7.2621e-02, -3.2584e-02,  5.2341e-02,  4.5055e-02,  8.2530e-03,
          3.6702e-02, -1.3942e-02,  6.5392e-02, -2.6427e-02,  2.0638e-04,
         -1.3664e-02, -3.6281e-02, -1.9504e-02, -2.8974e-02,  3.9427e-02,
         -8.8409e-02,  2.6243e-03,  1.3671e-02,  4.8306e-02, -3.1157e-02,
         -1.1733e-01, -5.1169e-02, -8.8529e-02, -2.1896e-02,  1.4299e-02,
          4.4417e-02, -1.3482e-02,  7.4339e-02,  2.6638e-02, -1.9876e-02,
          1.7919e-02, -1.0605e-02, -9.0426e-02,  2.1327e-02,  1.4120e-01,
         -6.4717e-03, -1.4039e-03, -1.

In [12]:
import torch
import numpy as np
from torch import Tensor
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from beir import util as beir_util
from beir.datasets.data_loader import GenericDataLoader
import os
import logging
from collections import defaultdict
from custum_evals import MultiGPUInformationRetrievalEvaluator, UBinarySentenceTransformer, hamming_similarity_from_distance


from datasets import load_dataset


def get_dataset(dataset_path= "nasa-impact/nasa-sde-IR-benchmark-sample-v2"):

    corpus = load_dataset(
        dataset_path,
        data_files="corpus.jsonl",
        split="train",
        token=os.environ['HUGGINGFACE_TOKEN']
    )
    queries = load_dataset(
        dataset_path,
        data_files="queries.jsonl",
        split="train",
        token=os.environ['HUGGINGFACE_TOKEN']
    )
    relevant_docs_data = load_dataset(
        dataset_path,
        split="test",
        data_files=None,
        token=os.environ['HUGGINGFACE_TOKEN']
    )

    corpus = {row["_id"]: row["text"] for i, row in enumerate(corpus)}
    queries = {row["_id"]: row["text"] for row in queries}
    relevant_docs_data = (
        relevant_docs_data.to_pandas()
        .groupby("query-id")["corpus-id"]
        .apply(set)
        .to_dict()
    )
    relevant_docs_data = {
        str(k): {str(item) for item in v} for k, v in relevant_docs_data.items()
    }

    return corpus, queries, relevant_docs_data

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


# --- Main Evaluation Script ---

def main():
    """
    Main function to run the comparison between standard and binary embeddings.
    """

    # Initialize Models ---
    model_name = '/rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000'
    standard_model = SentenceTransformer(model_name)
    binary_model = UBinarySentenceTransformer(model_name)
    logging.info(f"Models '{model_name}' (standard and binary) initialized.")


    corpus, queries, relevant_docs = get_dataset()

    # 4. --- Perform Retrieval and Evaluation ---
    
    # -- 4a. Evaluate Standard Model (Cosine Similarity) --
    logging.info("Evaluating standard model performance with InformationRetrievalEvaluator...")
    standard_evaluator = MultiGPUInformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name="standard-eval")
    scores_standard = standard_evaluator(standard_model)

    # -- 4b. Evaluate Binary Model (Hamming Similarity) --
    logging.info("Evaluating binary model performance with InformationRetrievalEvaluator...")
    # We create a new evaluator that knows how to use our custom hamming similarity function.
    binary_evaluator = MultiGPUInformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        score_functions={"hamming": hamming_similarity_from_distance},
        name='binary-eval'
    )
    scores_binary = binary_evaluator(binary_model)

    # 5. --- Print Comparison ---
    print("\n\n" + "="*50)
    print("      Information Retrieval Performance Comparison")
    print("="*50)
    # print(f"Dataset: {dataset}")
    print(f"Model: {model_name}")
    print(f"Queries Evaluated: {len(queries)}")
    print("-"*50)
    print("Metric       | Standard (Float32) | Binary Quantized")
    print("-"*50)

    # Dynamically find the keys for the metrics, as they are prefixed by the score function name.
    ndcg_key_standard = next((key for key in scores_standard if 'ndcg@10' in key.lower()), None)
    mrr_key_standard = next((key for key in scores_standard if 'mrr@10' in key.lower()), None)
    ndcg_key_binary = next((key for key in scores_binary if 'ndcg@10' in key.lower()), None)
    mrr_key_binary = next((key for key in scores_binary if 'mrr@10' in key.lower()), None)

    ndcg_standard = scores_standard.get(ndcg_key_standard, 0)
    mrr_standard = scores_standard.get(mrr_key_standard, 0)
    ndcg_binary = scores_binary.get(ndcg_key_binary, 0)
    mrr_binary = scores_binary.get(mrr_key_binary, 0)
    
    print(f"NDCG@10      | {ndcg_standard:.4f}             | {ndcg_binary:.4f}")
    print(f"MRR@10       | {mrr_standard:.4f}             | {mrr_binary:.4f}")
    print("="*50)


if __name__ == "__main__":
    main()

2025-08-28 16:42:47,009 - INFO - Use pytorch device_name: cuda:0
2025-08-28 16:42:47,010 - INFO - Load pretrained SentenceTransformer: /rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000
2025-08-28 16:42:47,617 - INFO - Use pytorch device_name: cuda:0
2025-08-28 16:42:47,618 - INFO - Load pretrained UBinarySentenceTransformer: /rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000
2025-08-28 16:42:47,619 - WARNING - No sentence-transformers model found with name /rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000. Creating a new one with mean pooling.
2025-08-28 16:42:47,827 - INFO - Models '/rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000' (standard and binary) initialized.
2025-08-28 16:42:51,247 - INFO - Evaluating standard model performance with InformationRetrievalEvaluator...
2025-08-28 16:4

Computing query embeddings


Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

2025-08-28 16:43:19,235 - INFO - Computing embeddings for corpus


Computing corpus embeddings


...

2025-08-28 16:58:38,552 - INFO - Computing embeddings for corpus


Computing corpus embeddings


Chunks:   0%|          | 0/63 [00:00<?, ?it/s]

2025-08-28 17:13:48,399 - INFO - Queries: 1500
2025-08-28 17:13:48,400 - INFO - Corpus: 63885

2025-08-28 17:13:48,458 - INFO - Score-Function: hamming
2025-08-28 17:13:48,458 - INFO - Accuracy@1: 30.73%
2025-08-28 17:13:48,458 - INFO - Accuracy@3: 45.67%
2025-08-28 17:13:48,458 - INFO - Accuracy@5: 51.80%
2025-08-28 17:13:48,459 - INFO - Accuracy@10: 60.60%
2025-08-28 17:13:48,459 - INFO - Precision@1: 30.73%
2025-08-28 17:13:48,459 - INFO - Precision@3: 15.22%
2025-08-28 17:13:48,459 - INFO - Precision@5: 10.36%
2025-08-28 17:13:48,460 - INFO - Precision@10: 6.06%
2025-08-28 17:13:48,460 - INFO - Recall@1: 30.73%
2025-08-28 17:13:48,460 - INFO - Recall@3: 45.67%
2025-08-28 17:13:48,460 - INFO - Recall@5: 51.80%
2025-08-28 17:13:48,460 - INFO - Recall@10: 60.60%
2025-08-28 17:13:48,461 - INFO - MRR@10: 0.3988
2025-08-28 17:13:48,461 - INFO - NDCG@10: 0.4483
2025-08-28 17:13:48,461 - INFO - MAP@100: 0.4089




      Information Retrieval Performance Comparison
Model: /rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000
Queries Evaluated: 1500
--------------------------------------------------
Metric       | Standard (Float32) | Binary Quantized
--------------------------------------------------
NDCG@10      | 0.4983             | 0.4483
MRR@10       | 0.4500             | 0.3988
